# **Фантомизировать Сливер /PhantomizeSliver**
> Gemini를 활용한 Sliver Implant의 AV/EDR 등 정적 탐지 솔루션 우회 자동화
---
**학습 목표로 제작된 프로젝트로써,100% 정확한 정보를 제공하지 않을 수 있음.**

 **제공했을 때의 악용 가능성이 보이는 부분은 삭제하였으나, 이를 이용한 해킹 공격으로 생기는 법적인 책임은 모두 자신에게 있으며, 프로젝트 관리자는 이에 어떠한 책임도 지지 않음을 사전에 명시함.**

---
*   셀 순서대로 진행
*   강조하는 셀의 ChangeMe 수정 필요
*   출력된 코드는 드래그 후 편집기에서 저장하여 사용




# 처음 실행 시 필요 라이브러리 설치

In [ ]:
!pip install requests pygithub plyara yara-python google-genai pandas

import re
import os
import json
import requests
from pathlib import Path
from plyara import Plyara
import pandas as pd
from google import genai
from google.genai import types

# **GCP 인증**

> **!! CHANGE ME !!** *PROJECT_ID*와 *LOCATION*, *GEMINI_API_KEY*를 올바르게 변경



In [ ]:
from google.colab import auth
auth.authenticate_user()

import os
# !! CHANGE ME !!
PROJECT_ID = ""
LOCATION = ""
os.environ["GOOGLE_CLOUD_PROJECT"] = PROJECT_ID
os.environ['GEMINI_API_KEY'] = ''

# **Download YARA Rule**

> **!! CHANGE ME !!**


*   본인이 찾은 깃허브 raw url을 raw_urls 부분에 삽입.
*   기본적으로 YARA Rule 관련 정보는 미제공.




In [ ]:
import requests
from pathlib import Path

raw_urls = [
    # !! CHANGE ME !!
]

out_dir = Path("yara_rules")
out_dir.mkdir(exist_ok=True)

for url in raw_urls:
    r = requests.get(url, timeout=30)
    if r.status_code == 200:
        fname = out_dir / Path(url).name
        fname.write_text(r.text, encoding="utf-8")
        print("[+] Saved", fname)
    else:
        print("[-] Failed:", url, r.status_code)


# Parsing Yara Rules










In [ ]:
from pathlib import Path
import re
import pandas as pd

rules = []

rule_pattern = re.compile(
    r'rule\s+([A-Za-z0-9_\-]+).*?strings:(.*?)condition:',
    re.S | re.M
)
string_pattern = re.compile(
    r'([$\w\d_]+)\s*=\s*(["\'].*?["\']|\{[^\}]+\}|\/.*?\/)',
    re.S | re.M
)

for f in Path("yara_rules").glob("*.yar*"):
    try:
        content = f.read_text(encoding="utf-8", errors="ignore")
        matches = rule_pattern.findall(content)
        for rule_name, strings_block in matches:
            strings = []
            for sid, value in string_pattern.findall(strings_block):
                if value.strip().startswith("{"):
                    typ = "hex"
                elif value.strip().startswith("/"):
                    typ = "regex"
                else:
                    typ = "text"
                strings.append({"id": sid, "type": typ, "value": value.strip()})

            rules.append({
                "filename": str(f),
                "rule_name": rule_name,
                "strings": strings
            })
    except Exception as e:
        print("[-] parse error:", f, "-", e)

print(f"[+] Parsed {len(rules)} rules")

# Preview
if len(rules) > 0:
    df = pd.DataFrame(rules)
    display(df[["filename", "rule_name"]].head())
else:
    print("[!] No rules parsed. Check yara_rules directory path or file syntax.")



# 문자열 추출

In [ ]:
import re

def extract_all_literals(strings_list):
    extracted = []

    for s in strings_list:
        if not s:
            continue

        if isinstance(s, dict):
            val = s.get("value") or s.get("raw") or s.get("text") or s.get("string")
            typ = s.get("type", "text")
        else:
            val = str(s)
            typ = "text"

        if not val or not isinstance(val, str):
            continue

        val = val.strip()

        if typ.lower() == "hex" or re.match(r'^\{\s*([0-9A-Fa-f\?]{2}\s*)+\}$', val):
            extracted.append({"type": "hex", "raw": val})
            continue

        if typ.lower() == "regex" or re.match(r'^/.*/[imsxRU]*$', val) or (val.startswith('/') and val.endswith('/')):
            extracted.append({"type": "regex", "raw": val})
            continue

        m = re.search(r'["\']([^"\']+)["\']', val)
        if m:
            val = m.group(1).strip()

        extracted.append({"type": "text", "raw": val})

    return extracted

all_strings_raw = []
for r in rules:
    if not r or "strings" not in r:
        continue
    all_strings_raw.extend(extract_all_literals(r["strings"]))

unique_strings_map = {}
for s in all_strings_raw:
    raw_val = s["raw"]
    if not raw_val:
        continue
    unique_strings_map[raw_val] = s

unique_extracted_strings_all = list(unique_strings_map.values())
unique_extracted_strings_text_only = [
    s for s in unique_extracted_strings_all if s["type"] == "text"
]

unique_extracted_strings = unique_extracted_strings_text_only

print(f"[+] Extracted {len(unique_extracted_strings_all)} unique literals (All Types).")
print(f"[+] Filtered {len(unique_extracted_strings)} unique literals (Text Only).")

if len(unique_extracted_strings) == 0:
    print("[!]Warn: Text 문자열이 추출되지 못했습니다. Gemini는 빈 REPLACEMENTS 딕셔너리를 생성합니다.")



In [ ]:
## 추출 확인용
try:
    print(f"변수 유형: {type(unique_extracted_strings)}")
    print(f"항목 개수: {len(unique_extracted_strings)}")
    print(f"첫 5개 항목: {unique_extracted_strings[:5]}")
except NameError:
    print("[-] 오류: 'unique_extracted_strings' 변수가 정의되지 않았습니다.")

# **Gemini 호출**


$ORIGINAL_REPLACEMENTS는 AI에게 정보를 전달하기 위한 입력 매개변수로써, 필수적으로 채워야 함.

( 기본적으로 미제공. )



In [ ]:
ORIGINAL_REPLACEMENTS = {
    # !! CHANGE ME !!
}

def generate_modified_PhantomizeSliver_code(extracted_strings_info: list, original_replacements: dict) -> str:

    try:
        from google import genai
        from google.genai import types
    except ImportError:
        return "[-] Error: Google GenAI SDK가 설치되지 않았습니다. !pip install google-genai 로 해결하세요."

    try:
        client = genai.Client()
    except Exception as e:
        return f"Error: Gemini [-] 클라이언트 초기화 실패. GEMINI_API_KEY 환경 변수설정이 잘 되어있나 확인 바랍니다. 상세내용: {e}"

    original_replacements_str = json.dumps(original_replacements, indent=4)
    extracted_strings_str = "\n".join(
        f"- Type: {s.get('type')}, Raw: {s.get('raw')}" for s in extracted_strings_info
    )

    system_instruction = (
        "You are a cybersecurity expert specialized in malware analysis and evasion techniques. "
        "Your task is to generate a complete Python script named **PhantomizeSliver.py** that is functionally identical to the provided original script. "
        "The most critical part is the **REPLACEMENTS** dictionary. "
        "You must follow these instructions: "
        "1. **Think in English** and output the final result translated into Korean. "
        "2. Keep all functions (`rename_files_beacon`, `replace_string_IOCs`, `patch_donut`, `patch_max_connection`, `patch_dll_iocs`, `main`) and global variables (`BASE_DIR`, `TARGET_FILE`, `DONUT_FILE`, `EXCLUDE_DIRS`) **exactly** as they are in the original script. "
        "3. **Replace** the existing `REPLACEMENTS` dictionary with a **new one**. The new dictionary **MUST ONLY use strings of type 'text' (excluding 'hex' and 'regex')** from the extracted data as keys. Map each of these 'text' strings to a new, unique, and non-obvious replacement string to bypass YARA rule detection. "
        "4. Only output the final, complete Python code block (```python...```) and the required Korean translation instruction. No other text, explanation, or commentary is allowed."
    )

    prompt = (
        f"Generate the complete `PhantomizeSliver.py` Python script. "
        f"Use the following data:\n\n"
        f"### Original REPLACEMENTS Dictionary (for reference only):\n"
        f"```json\n{original_replacements_str}\n```\n\n"
        f"### New YARA Rule Strings to be Replaced:\n"
        f"The new **REPLACEMENTS** dictionary in `PhantomizeSliver.py` must use the following `Raw` strings (where type is 'text') as keys:\n"
        f"{extracted_strings_str}\n\n"
        f"Generate the complete, updated `PhantomizeSliver.py` script."
    )

    generation_config = types.GenerateContentConfig(
        system_instruction=system_instruction,
        temperature=0.8,
    )

    model_name = "gemini-2.5-pro"

    try:
        response = client.models.generate_content(
            model=model_name,
            contents=prompt,
            config=generation_config
        )
        return response.text
    except Exception as e:
        return f"[-] Error during Gemini API call: {e}"



# **코드 출력**

In [ ]:
print("--- Gemini API Call & Started generate PhantomizeSliver.py ---")
generated_code_korean = generate_modified_PhantomizeSliver_code(unique_extracted_strings, ORIGINAL_REPLACEMENTS)
print(generated_code_korean)

print("\n--- [+]Complete! ---")